# 13 — The Bures Bridge

Watch one formula change identity. The matrix term of the Gaussian Wasserstein distance, fed density matrices instead of covariances, *becomes* the quantum Bures distance of `02/12`. This is the door into quantum optimal transport.

**Prerequisites:** `12_bures_wasserstein`, `02/12_bures_distance`.

**What you'll be able to do**
- Show that the Bures–Wasserstein matrix term equals the squared quantum Bures distance on unit-trace matrices.
- Verify it on qubit states, including the $|+\rangle$ vs $I/2$ coherence case.
- Explain how "covariance $\to$ density matrix" defines a quantum Wasserstein and opens Module 4.

This is the brick the whole module has been building toward. The matrix term in the Gaussian Wasserstein formula is pure linear algebra on covariance matrices. Feed it *density matrices* instead, and it turns into the quantum Bures distance you met in `02/12`. Classical optimal transport and quantum state geometry meet in a single expression — and through that door is Module 4.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qot_course import viz
from qot_course.transport.gaussian import bures_matrix_distance
from qot_course.infotheory.quantum import bures_distance as quantum_bures_distance
from qot_course.quantum.density import density_matrix, maximally_mixed
from qot_course.quantum.states import KET_PLUS, KET_0, KET_1

np.random.seed(42)
viz.use_course_style()

## 1. The matrix term is the Bures distance

The covariance part of the Bures–Wasserstein formula,

$$ \mathrm{tr}(\Sigma_0) + \mathrm{tr}(\Sigma_1) - 2\,\mathrm{tr}\sqrt{\Sigma_0^{1/2}\,\Sigma_1\,\Sigma_0^{1/2}}, $$

is defined for **any** pair of Hermitian PSD matrices, not only covariances. Restrict it to **unit-trace** matrices — that is, density matrices $\rho_0, \rho_1$ — and the traces become $1$:

$$ 1 + 1 - 2\,\underbrace{\mathrm{tr}\sqrt{\sqrt{\rho_0}\,\rho_1\,\sqrt{\rho_0}}}_{F_U(\rho_0,\rho_1)\ \text{Uhlmann fidelity}} = 2\,(1 - F_U) = d_B^2(\rho_0, \rho_1). $$

That is *exactly* the squared Bures distance from `02/12`. So `bures_matrix_distance` on density matrices equals the quantum Bures distance squared. Let's check it on three qubit pairs.

In [ ]:
pairs = [
    ("|+> vs I/2",   density_matrix(KET_PLUS), maximally_mixed(2)),
    ("|0> vs |+>",   density_matrix(KET_0),    density_matrix(KET_PLUS)),
    ("|0> vs |1>",   density_matrix(KET_0),    density_matrix(KET_1)),
]

print(f"{'pair':<14s} {'d_B (02/12)':>12s} {'sqrt(BW matrix)':>16s} {'match?':>8s}")
print("-" * 54)
for label, rho, sigma in pairs:
    d_b = quantum_bures_distance(rho, sigma)
    sqrt_bw = float(np.sqrt(max(0.0, bures_matrix_distance(rho, sigma))))
    print(f"{label:<14s} {d_b:>12.4f} {sqrt_bw:>16.4f} {str(bool(np.isclose(d_b, sqrt_bw, atol=1e-6))):>8s}")

**Read the table.** For all three pairs — including the classic $|+\rangle$ vs $I/2$ case — the quantum Bures distance from `02/12` equals the square root of the Bures–Wasserstein matrix term to machine precision. The two formulas are one object seen from two sides: on **classical Gaussians** it gives the Wasserstein-2 distance; on **quantum density matrices** it gives the Bures distance.

In [ ]:
labels = [p[0] for p in pairs]
d_b_vals = [quantum_bures_distance(rho, sigma) for _, rho, sigma in pairs]
sqrt_bw_vals = [float(np.sqrt(max(0.0, bures_matrix_distance(rho, sigma)))) for _, rho, sigma in pairs]

x = np.arange(len(pairs))
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(x - 0.18, d_b_vals, width=0.36, color=viz.COLORS["quantum"], label="quantum Bures  d_B  (02/12)")
ax.bar(x + 0.18, sqrt_bw_vals, width=0.36, color=viz.COLORS["flow"], label="sqrt(Bures-Wasserstein matrix term)  (here)")
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel("distance")
ax.set_title("Same number, two derivations: quantum Bures = sqrt(BW matrix term)", pad=12)
ax.legend()
plt.show()

**Read the figure.** Each pair of bars has equal height — the two derivations give the same distance. The $|0\rangle$ vs $|1\rangle$ pair (orthogonal states) reaches the maximum $\sqrt{2}$. The $|+\rangle$ vs $I/2$ pair is the telling one: those two states share the *same* $Z$-diagonal $(\tfrac12, \tfrac12)$, yet their Bures distance is strictly positive — because Bures sees the off-diagonal **coherence** of $|+\rangle$ that a diagonal-only, classical view would miss. This is the `01/06` and `02/12` punchline, now living inside a transport formula.

## 2. What the bridge means

Read the equality the other way. The Gaussian story gave a Wasserstein distance from a covariance matrix; replace the covariance by a **density matrix** and the very same formula returns a distance between *quantum states*. That is the definition Module 4 will make precise:

- lift the LP marginals from mass vectors $a, b$ to density matrices $\rho_A, \rho_B$;
- the Bures–Wasserstein matrix expression sits naturally inside the resulting semidefinite program;
- the Bures distance is then the **quantum Wasserstein on commuting states** — a special case we can already compute.

So the Bures distance is one of the **three faces of quantum optimal transport** (the Bures-bridge object). The other two — a coupling SDP and an entropic/quantum-Sinkhorn formulation — appear in Module 4, and a later notebook reconciles all three. You have built the first.

## Your turn

1. **Mean-only Gaussians, again.** Take two Gaussians with the same mean and unit-trace covariances $\rho_0, \rho_1$. Confirm $W_2 = \sqrt{\texttt{bures\_matrix\_distance}(\rho_0, \rho_1)}$ equals `quantum_bures_distance` — the classical and quantum readings of one number.
2. **Coherence drives the distance.** Interpolate $\rho_\theta = \cos^2\theta\,|+\rangle\langle+| + \sin^2\theta\,I/2$ and plot its Bures distance to $I/2$ as $\theta$ varies. Where is it largest, and why does the $Z$-diagonal stay fixed throughout?
3. **Orthogonality is the ceiling.** Confirm that the Bures distance between two orthogonal pure states is $\sqrt{2}$, and argue from $2(1 - F_U)$ why no pair of states can exceed it.

## What you built

- You showed the Bures–Wasserstein matrix term equals the squared quantum Bures distance on unit-trace matrices.
- You verified it on qubit pairs, and saw $|+\rangle$ vs $I/2$ stay positive despite an identical $Z$-diagonal — Bures sees coherence.
- You can now explain how "covariance $\to$ density matrix" turns a classical transport formula into a quantum one.

You have built the bridge the course is named for. One formula, two worlds: classical Wasserstein and quantum Bures are the same expression, and that is what makes quantum optimal transport possible at all.

## What's next

`14_benamou_brenier_otto` closes Module 3 by viewing the $W_2$ geodesic two ways — McCann's affine map on Gaussians, and Benamou–Brenier's least-energy flow (Otto's Riemannian metric) — and completes the dictionary before Module 4 lifts everything to quantum states.

## References

- R. Bhatia, T. Jain, Y. Lim, "On the Bures–Wasserstein distance between positive definite matrices", *Expo. Math.* 37, 165–191 (2019). DOI:10.1016/j.exmath.2018.01.002
- A. Uhlmann, "The 'transition probability' in the state space of a *-algebra", *Rep. Math. Phys.* 9, 273–279 (1976).
- D. Bures, "An extension of Kakutani's theorem on infinite product measures to the tensor product of semifinite w*-algebras", *Trans. AMS* 135, 199–212 (1969).

**Previous:** `notebooks/03_ClassicalOptimalTransport/12_bures_wasserstein.ipynb` · **Next:** `notebooks/03_ClassicalOptimalTransport/14_benamou_brenier_otto.ipynb`